# Домашнее задание 7. Построение систем текстового поиска

В этом ДЗ вы соберёте **мини-поисковую систему** под e-commerce сценарий:  
**полнотекстовый BM25 → векторный поиск → гибрид + простое переранжирование**.

**Контекст (приближён к продакшену):** есть логи взаимодействий пользователя с выдачей (у нас это `esci_label`), и хочется использовать их как *поведенческий сигнал* — но **без таргет-лика**.

---

## Что в итоге должно получиться
1) **BM25-поиск** по текстам товаров + корректный расчёт метрики качества.  
2) **Векторный поиск** (эмбеддер → индекс → выдача).  
3) **Гибридный поиск**: объединяем кандидатов и учим простой **reranker**.

## Данные и полезные ссылки
- Датасет: **Amazon ESCI** на Hugging Face — https://huggingface.co/datasets/tasksource/esci  
- Docker Desktop (нужно для бонусного трека с OpenSearch/Qdrant): https://docs.docker.com/desktop/  
- OpenSearch:
  - quickstart на Python: https://last9.io/blog/how-to-use-opensearch-with-python/
  - types в mappings: https://docs.opensearch.org/docs/latest/field-types/supported-field-types/index/  
  - настройка BM25 similarity: https://docs.opensearch.org/docs/latest/search-plugins/keyword-search/#configuring-bm25-similarity  
  - match query (Query DSL): https://docs.opensearch.org/docs/latest/query-dsl/full-text/match/  
  - function_score (сложные запросы): https://cwillum.github.io/query-dsl/compound/function-score/  
- Qdrant quickstart: https://qdrant.tech/documentation/quickstart/  

> В основной (Colab-friendly) версии мы используем лёгкую реализацию BM25 и embedded Qdrant,  
> а **бонусный трек** позволяет повторить часть лекции с **OpenSearch Query DSL**.

---

## Связь с лекцией
- Обратный индекс и логика поиска: **слайды 16–18**  
- Метрики ранжирования (DCG/nDCG, mAP): **слайды 55–60**  
- BM25 и его интуиция (TF/IDF, параметры): **слайды 21–30**  
- Векторный поиск (плюсы/минусы): **слайды 35–37**  
- Гибридный поиск и компромиссы: **слайды 41-45**

---

## Подсказка по данным
Мы **сознательно берём маленький срез** датасета (настройки в `DATA_CFG`), чтобы ноутбук был быстрым и дружелюбным к Colab.


## Подготовка в Google Colab

1) Откройте ноутбук в Colab: **File → Upload notebook** (или откройте по ссылке от платформы).  
2) Рекомендуемый runtime:
   - **CPU** хватит, но эмбеддинги будут считаться дольше;
   - если есть возможность — включите **GPU**: *Runtime → Change runtime type → GPU*.
3) Нажмите **Runtime → Run all** и дождитесь, пока пройдёт блок *«Установка окружения»* и *«Загрузка данных»*.  
4) Дальше выполнение остановится на местах `# ---- Ваш код здесь ----`.  
   Заполняйте их по порядку: тесты под каждой задачей быстро покажут, что вы всё подключили корректно.

Если что-то «сломалось» (например, после переустановки пакетов):
- **Runtime → Restart runtime**, затем снова *Run all*.


### Если хотите бонусный трек с OpenSearch (как в лекции)
- В Colab **Docker может не запуститься** (зависит от окружения). Тогда бонус лучше делать **локально**.
- Локально: поставьте Docker и выполните команды из секции **«Бонусный трек: OpenSearch»** ниже.


In [1]:
# Установка окружения
# Важно: если вы в Colab — после установки пакетов имеет смысл сделать Restart Runtime.
%pip install -U -q tensorflow[and-cuda]

%pip -q install \
  datasets==2.20.0 \
  rank-bm25==0.2.2 \
  qdrant-client==1.10.1 \
  sentence-transformers==3.0.1 \
  catboost==1.2.8 \
  pandas==2.2.2 \
  tqdm==4.66.4

import os
import re
import math
import random
import typing as T
from dataclasses import dataclass
from collections import defaultdict

import numpy as np
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


Note: you may need to restart the kernel to use updated packages.


ERROR: Cannot install tensorflow[and-cuda]==2.20.0 and tensorflow[and-cuda]==2.21.0 because these package versions have conflicting dependencies.

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Загрузка данных и вспомогательные функции

from datasets import load_dataset

DATA_CFG = {
    # Сколько «сырых» строк взять из датасета (чем меньше, тем быстрее)
    "train_slice": "train[:20000]",
    "test_slice": "test[:5000]",
    # Сколько запросов оставить в каждой части (после дедупликации)
    "n_indexed_queries": 400,  # используем только для поведенческих фичей
    "n_train_queries": 400,    # обучаем reranker
    "n_test_queries": 250,     # честно меряем качество
}

LABEL2GAIN = {
    "Exact": 3,
    "Substitute": 2,
    "Complement": 1,
    # Остальные/неизвестные будем считать нерелевантными
}

@dataclass(frozen=True)
class TextFields:
    # Упрощаем вход: берём только 2 поля, с которыми реально работаем в выдаче
    product_title: str
    product_text: str

@dataclass
class DatasetPart:
    queries: dict[str, str]                       # query_id -> query_text
    query2doc2target: dict[str, dict[str, str]]   # query_id -> {product_id -> label}

def _normalize_text(x: str) -> str:
    x = x.lower().strip()
    x = re.sub(r"\s+", " ", x)
    return x

def tokenize(text: str) -> list[str]:
    # Простая токенизация для BM25 (в лекции обсуждается, что это отдельный модуль пайплайна)
    text = _normalize_text(text)
    return re.findall(r"\w+", text, flags=re.UNICODE)

def process_hf_split(split) -> tuple[dict[str, TextFields], DatasetPart]:
    """Превращаем HF split в удобные структуры.

    Важно: в датасете есть повторяющиеся записи (example_id), удаляем дубликаты.
    """
    seen_example_ids = set()
    docs: dict[str, TextFields] = {}
    queries: dict[str, str] = {}
    query2doc2target: dict[str, dict[str, str]] = defaultdict(dict)

    for row in tqdm(split, desc="process"):
        ex_id = row["example_id"]
        if ex_id in seen_example_ids:
            continue
        seen_example_ids.add(ex_id)

        doc_id = row["product_id"]
        if doc_id not in docs:
            docs[doc_id] = TextFields(
                product_title=row["product_title"],
                product_text=row["product_text"],
            )

        q_id = row["query_id"]
        if q_id not in queries:
            queries[q_id] = row["query"]

        query2doc2target[q_id][doc_id] = row["esci_label"]

    return docs, DatasetPart(queries=queries, query2doc2target=dict(query2doc2target))

def _sample_dict(d: dict[str, T.Any], n: int, seed: int = SEED) -> dict[str, T.Any]:
    keys = list(d.keys())
    rnd = random.Random(seed)
    rnd.shuffle(keys)
    keys = keys[:min(n, len(keys))]
    return {k: d[k] for k in keys}

def _subsample_part(part: DatasetPart, n_queries: int, seed: int = SEED) -> DatasetPart:
    q = _sample_dict(part.queries, n_queries, seed=seed)
    q2d2t = {qid: part.query2doc2target[qid] for qid in q}
    return DatasetPart(queries=q, query2doc2target=q2d2t)

# 1) Загружаем маленький срез (быстро в Colab)
raw_train = load_dataset("tasksource/esci", split=DATA_CFG["train_slice"])
raw_test = load_dataset("tasksource/esci", split=DATA_CFG["test_slice"])

# 2) Преобразуем в удобные структуры
train_docs, train_part_full = process_hf_split(raw_train)
test_docs, test_part_full = process_hf_split(raw_test)

# 3) Собираем общий каталог документов (как если бы это был общий каталог товаров)
text_rich_docs = dict(train_docs)
text_rich_docs.update(test_docs)

# 4) Делаем финальные сплиты по запросам:
#    indexed — только для снятия поведенческих фичей (не используем в обучении!)
#    train   — обучение reranker
#    test    — финальная оценка
half = len(train_part_full.queries) // 2
train_q_ids = list(train_part_full.queries.keys())
random.Random(SEED).shuffle(train_q_ids)

indexed_q_ids = set(train_q_ids[:half])
train_q_ids = set(train_q_ids[half:])

indexed_part = DatasetPart(
    queries={qid: train_part_full.queries[qid] for qid in indexed_q_ids},
    query2doc2target={qid: train_part_full.query2doc2target[qid] for qid in indexed_q_ids},
)
train_part = DatasetPart(
    queries={qid: train_part_full.queries[qid] for qid in train_q_ids},
    query2doc2target={qid: train_part_full.query2doc2target[qid] for qid in train_q_ids},
)
test_part = test_part_full

# 5) Дополнительно уменьшаем части (чтобы всё было быстрым)
indexed_part = _subsample_part(indexed_part, DATA_CFG["n_indexed_queries"], seed=SEED)
train_part = _subsample_part(train_part, DATA_CFG["n_train_queries"], seed=SEED + 1)
test_part = _subsample_part(test_part, DATA_CFG["n_test_queries"], seed=SEED + 2)

def part_stats(part: DatasetPart) -> dict[str, int]:
    n_points = sum(len(v) for v in part.query2doc2target.values())
    n_docs = len({doc for d2t in part.query2doc2target.values() for doc in d2t})
    return {"queries": len(part.queries), "docs": n_docs, "points": n_points}

print("indexed:", part_stats(indexed_part))
print("train  :", part_stats(train_part))
print("test   :", part_stats(test_part))


Generating train split:   0%|          | 0/2027874 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/652490 [00:00<?, ? examples/s]

process:   0%|          | 0/20000 [00:00<?, ?it/s]

process:   0%|          | 0/5000 [00:00<?, ?it/s]

indexed: {'queries': 400, 'docs': 933, 'points': 1183}
train  : {'queries': 400, 'docs': 587, 'points': 808}
test   : {'queries': 250, 'docs': 655, 'points': 808}


## Часть 1. Полнотекстовый поиск (BM25)

Здесь мы строим полнотекстовый ранжирующий retrieval-канал. В проде эту часть часто делают в OpenSearch/Elasticsearch/Solr, но в домашке достаточно компактного BM25, чтобы отработать принципы.

# Задача 1: Покрытие документов и риск таргет-лика – 2 балла

**Зачем:** перед тем как «подмешивать» поведенческие признаки в ранжирование, важно понять,
насколько каталог документов из `indexed` покрывает `train/test`.  
Это напрямую связано с идеей про **разделение данных** и «не подсматривать в будущее».

**Связь с лекцией:** постановка задачи поиска и архитектура пайплайна (**слайды 11–12**).

### Что нужно сделать
Реализуйте `compute_coverage(...)`, которая считает:
- сколько уникальных документов встречается в `indexed/train/test`;
- сколько документов из `indexed` встречается в `train`;
- сколько документов из `indexed ∪ train` встречается в `test`.

Возвращайте всё в одном словаре.


In [3]:
# ---- Ваш код здесь ----
def compute_coverage(indexed: DatasetPart, train: DatasetPart, test: DatasetPart) -> dict[str, int]:
    '''
    Returns dict with keys:
      - n_docs_indexed, n_docs_train, n_docs_test
      - overlap_indexed_train
      - overlap_indexed_train__test   (docs in (indexed ∪ train) ∩ test)
    '''
    # YOUR CODE
    indexed_docs = {
        doc_id for docs in indexed.query2doc2target.values() for doc_id in docs
    }
    train_docs = {
        doc_id for docs in train.query2doc2target.values() for doc_id in docs
    }
    test_docs = {
        doc_id for docs in test.query2doc2target.values() for doc_id in docs
    }

    return {
        "n_docs_indexed": len(indexed_docs),
        "n_docs_train": len(train_docs),
        "n_docs_test": len(test_docs),
        "overlap_indexed_train":  sum(
            doc_id in train_docs for doc_id in indexed_docs
        ),
        "overlap_indexed_train__test":  sum(
            doc_id in test_docs for doc_id in (indexed_docs | train_docs)
        ),
    }
# ---- Конец кода ----

In [4]:
# Ассерт-тест к задаче 1

toy_indexed = DatasetPart(
    queries={"q1": "a"},
    query2doc2target={"q1": {"d1": "Exact", "d2": "Complement"}},
)
toy_train = DatasetPart(
    queries={"q2": "b"},
    query2doc2target={"q2": {"d2": "Substitute", "d3": "Exact"}},
)
toy_test = DatasetPart(
    queries={"q3": "c"},
    query2doc2target={"q3": {"d3": "Complement", "d4": "Exact"}},
)

cov = compute_coverage(toy_indexed, toy_train, toy_test)

assert cov["n_docs_indexed"] == 2
assert cov["n_docs_train"] == 2
assert cov["n_docs_test"] == 2
assert cov["overlap_indexed_train"] == 1  # d2
assert cov["overlap_indexed_train__test"] == 1  # d3
print("✅ Task 1 OK")


✅ Task 1 OK


In [17]:
# Прогон на реальных сплитах (не оценивается, но полезно понимать цифры)

from rich import print as rprint
rprint(compute_coverage(indexed_part, train_part, test_part))

# надо глазами посмотретьЮ что там реально лежит
for name, part in [
    ("indexed", indexed_part),
    ("train", train_part),
    ("test", test_part),
]:
    query_id = next(iter(part.query2doc2target))
    docs = part.query2doc2target[query_id]
    doc_id = next(iter(docs))
    document = text_rich_docs[doc_id]

    print(f"\n--- {name} ---")
    print("query_id:", query_id)
    print("Запрос:", part.queries[query_id])
    print("doc_id:", doc_id)
    print("label:", docs[doc_id])
    print("Название:", document.product_title)
    print("Текст:", document.product_text)



{
    'n_docs_indexed': 933,
    'n_docs_train': 587,
    'n_docs_test': 655,
    'overlap_indexed_train': 234,
    'overlap_indexed_train__test': 258
}


--- indexed ---
query_id: 71700
Запрос: music stickers
doc_id: B07PQBJJL4
label: Exact
Название: Band Stickers 100 Pcs Rock and Roll Music Stickers, Vinyl Waterproof Stickers for Personalize Laptop, Electronic Organ, Guitar, Piano, Helmet, Skateboard, Luggage Graffiti Decals
Текст: Band Stickers 100 Pcs Rock and Roll Music Stickers, Vinyl Waterproof Stickers for Personalize Laptop, Electronic Organ, Guitar, Piano, Helmet, Skateboard, Luggage Graffiti Decals
TUYOART
None
<b>TUYOART</b> is a professional sticker brand, dedicated to create a diversified life for people.<br> <br> We have many different styles of stickers for you to choose from, whether you are a child, a teenager or an adult.<br> <br> Our sticker is made up of PVC, with high quality and the function of waterproof and sun-protection, you can feel free to post anywhere you like, such as a laptop, guitar, bike, etc.<br> <br> <b>Various Pictures</b><br> <br> The stickers is 100% new with high-definition printing. One package 

# Задача 2: Подокументные поведенческие признаки – 3 балла

**Зачем:** мы хотим «усилить» поиск сигналом из логов (как *boost* к скору).  
В лекции обсуждается, что ранжирование — это комбинация разных сигналов (текст + поведение + ...).

**Связь с лекцией:** пайплайн и идея «встроить сигнал в ранжирование» (**слайды 11–12**).

### Что нужно сделать
Реализуйте `obtain_behavior_features(part)`, которая строит отображение:
`product_id -> BehaviourFeatures`.

Подсказка: в `part.query2doc2target` уже есть (query_id → doc_id → label).  
Нужно аккуратно агрегировать по документу.


In [20]:
@dataclass
class BehaviourFeatures:
    # Счётчики по типам релевантности
    exact: float = 0.0
    complement: float = 0.0
    substitute: float = 0.0

    # Часто удобно иметь агрегированный счётчик «попаданий в целом»
    exact_or_complement: float = 0.0

    def __iter__(self):
        for k in self.__dict__:
            yield self.__dict__[k]

    def __add__(self, other: "BehaviourFeatures") -> "BehaviourFeatures":
        return BehaviourFeatures(*(a + b for a, b in zip(tuple(self), tuple(other))))

# ---- Ваш код здесь ----
def obtain_behavior_features(part: DatasetPart) -> dict[str, BehaviourFeatures]:
    '''
    Возвращает doc_id -> BehaviourFeatures.
    '''
    # YOUR CODE
    doc2features = {}

    for docs in part.query2doc2target.values():
        for doc_id, label in docs.items():
            if doc_id not in doc2features:
                doc2features[doc_id] = BehaviourFeatures()

            features = doc2features[doc_id]

            if label == "Exact":
                features.exact += 1                 # я только не понял, зачем они типа float
                features.exact_or_complement += 1   # и почему было просто не просуммировать
            elif label == "Complement":
                features.complement += 1
                features.exact_or_complement += 1
            elif label == "Substitute":
                features.substitute += 1

    return doc2features
# ---- Конец кода ----

In [21]:
# Ассерт-тест к задаче 2

toy = DatasetPart(
    queries={"q1": "a", "q2": "b"},
    query2doc2target={
        "q1": {"d1": "Exact", "d2": "Complement", "d3": "Substitute"},
        "q2": {"d1": "Complement", "d3": "Substitute"},
    },
)

doc2f = obtain_behavior_features(toy)
rprint(doc2f["d1"])

assert doc2f["d1"].exact == 1
assert doc2f["d1"].complement == 1
assert doc2f["d1"].exact_or_complement == 2

assert doc2f["d2"].complement == 1
assert doc2f["d2"].exact_or_complement == 1

assert doc2f["d3"].substitute == 2
assert doc2f["d3"].exact_or_complement == 0

print("✅ Task 2 OK")


BehaviourFeatures(exact=1.0, complement=1.0, substitute=0.0, exact_or_complement=2.0)

✅ Task 2 OK


In [22]:
# Снимаем поведенческие признаки (в проде это обычно идёт из логов / feature-store)
indexed_behaviour = obtain_behavior_features(indexed_part)
train_behaviour = obtain_behavior_features(train_part)

def merge_features(a: dict[str, BehaviourFeatures], b: dict[str, BehaviourFeatures]) -> dict[str, BehaviourFeatures]:
    out: dict[str, BehaviourFeatures] = defaultdict(BehaviourFeatures)
    for k, v in a.items():
        out[k] = out[k] + v
    for k, v in b.items():
        out[k] = out[k] + v
    return dict(out)

indexed_plus_train_behaviour = merge_features(indexed_behaviour, train_behaviour)

print("docs with behaviour:", len(indexed_plus_train_behaviour))


docs with behaviour: 1286


# Задача 3: Индексация BM25 – 5 баллов

**Зачем:** прежде чем «искать», нужно построить индекс и определить,
какие поля каталога участвуют в полнотекстовом ранжировании.

**Связь с лекцией:**
- обратный индекс (**слайды 21–23**),
- BM25 и TF/IDF (**слайды 27–39**).

### Что нужно сделать
1) Реализуйте `build_bm25_index(...)`:
   - соберите корпус документов (список `doc_ids`);
   - для каждого документа подготовьте текст (по выбранным полям);
   - токенизируйте;
   - создайте BM25-модель.

2) Индекс должен уметь отвечать на запросы через метод `.search(query_text, top_k)`.


> Если вы делаете **бонусный трек (OpenSearch)**, то вместо `build_bm25_index` вы будете собирать **mappings** и документ для индексации. Ссылки на документацию — в начале ноутбука.


In [23]:
from rank_bm25 import BM25Okapi

@dataclass
class BM25Index:
    doc_ids: list[str]
    bm25: BM25Okapi
    tokenized_corpus: list[list[str]]

    def search(self, query_text: str, top_k: int = 15) -> list[dict[str, T.Any]]:
        tokens = tokenize(query_text)
        scores = self.bm25.get_scores(tokens)
        # берём top_k индексов по убыванию скоринга
        best_idx = np.argsort(scores)[::-1][:top_k]
        return [{"doc_id": self.doc_ids[i], "bm25_score": float(scores[i])} for i in best_idx]


def build_doc_text(doc: TextFields, fields: list[str]) -> str:
    # Здесь можно добавлять более умные вещи (нормализация, буст title, etc.)
    chunks = []
    for f in fields:
        chunks.append(getattr(doc, f))
    return " ".join(chunks)

# ---- Ваш код здесь ----
def build_bm25_index(
    text_rich_docs: dict[str, TextFields],
    doc_ids: list[str],
    fields: list[str],
) -> BM25Index:
    '''
    Должен вернуть BM25Index.
    '''
    # YOUR CODE
    # вычленяем простой текст с заголовком, токенизируем и строим индекс BM25
    # для улучшения потом можно будет добавить нормализацию, бустинг заголовка и подумать что ещё
    tokenized_corpus = [
        tokenize(build_doc_text(text_rich_docs[doc_id], fields))
        for doc_id in doc_ids
    ]

    return BM25Index(
        doc_ids=doc_ids,
        bm25=BM25Okapi(tokenized_corpus),
        tokenized_corpus=tokenized_corpus,
    )
# ---- Конец кода ----

In [24]:
# Ассерт-тест к задаче 3

toy_docs = {
    "d1": TextFields(product_title="red shoes", product_text="comfortable running shoes"),
    "d2": TextFields(product_title="blue shirt", product_text="cotton t-shirt"),
    "d3": TextFields(product_title="running socks", product_text="socks for marathon"),
}
idx = build_bm25_index(toy_docs, doc_ids=["d1","d2","d3"], fields=["product_title","product_text"])
serp = idx.search("running shoes", top_k=2)
assert len(serp) == 2
assert serp[0]["doc_id"] in {"d1","d3"}  # оба релевантны по словам running/shoes/socks
assert all("bm25_score" in x for x in serp)
print("✅ Task 3 OK")


✅ Task 3 OK


In [25]:
# Строим полнотекстовый индекс на реальных данных

ALL_DOC_IDS = sorted({doc for part in [indexed_part, train_part, test_part] for d2t in part.query2doc2target.values() for doc in d2t})
print("Docs to index:", len(ALL_DOC_IDS))

BM25_FIELDS = ["product_title", "product_text"]  # можно поэкспериментировать, но оставим так по умолчанию
bm25_index = build_bm25_index(text_rich_docs, doc_ids=ALL_DOC_IDS, fields=BM25_FIELDS)


Docs to index: 1683


# Задача 4: Запрос, буст поведенческим сигналом и оценка качества – 10 баллов

**Зачем:** BM25 хорошо работает на точных совпадениях, но в проде часто хочется
добавить дополнительный сигнал (например, «популярность» или «частота кликов»).
Здесь мы добавим буст на основе `BehaviourFeatures.exact`.

**Связь с лекцией:**
- BM25 и его ограничения (**слайды 37–39**),
- метрики качества ранжирования (**слайды 13–16**).

### Что нужно сделать
1) Реализуйте `rerank_with_behaviour(...)` — функцию, которая пересчитывает скор:
   - берём BM25 score;
   - домножаем на монотонную функцию от поведенческого сигнала (пример в условии).
2) Реализуйте `evaluate_bm25(...)` и посчитайте средний `DCG@K` на `test_part`.


> В OpenSearch это обычно делается через **Query DSL**: `match/multi_match` + `function_score`.
Ссылки на примеры — в начале ноутбука.


In [35]:
def dcg_at_k(gains: list[float], k: int = 15) -> float:
    gains = np.asarray(gains[:k], dtype=float)
    if len(gains) == 0:
        return 0.0
    discounts = np.log2(np.arange(2, len(gains) + 2))
    return float(np.sum((2**gains - 1) / discounts))

def gains_for_query(serp_doc_ids: list[str], gt_doc2label: dict[str, str]) -> list[float]:
    gains = []
    for doc_id in serp_doc_ids:
        label = gt_doc2label.get(doc_id, None)
        gains.append(float(LABEL2GAIN.get(label, 0.0)))
    return gains

# ---- Ваш код здесь ----
def rerank_with_behaviour(
    bm25_serp: list[dict[str, T.Any]],
    doc2behaviour: dict[str, BehaviourFeatures],
    add: float = 1.0,
    mult: float = 2.0,
) -> list[dict[str, T.Any]]:
    '''
    На входе: список {'doc_id', 'bm25_score'}.
    На выходе: такой же список, но отсортированный по финальному скору.

    Идея (можно ровно так, можно эквивалентно):
      final = bm25_score * log(add + mult * exact + complement)
    '''
    # YOUR CODE
    reranked = []

    for item in bm25_serp:
        behaviour = doc2behaviour.get(item["doc_id"])

        # Без поведенческих данных сохраняем исходный скор. ФОрмула в "Идее" его обнуляла, мне это не кажется хороший идеей.
        # потом посмотрю по ходу задания, как это влияет на качество и тогда решим, что делать без поведенческих данных.
        factor = 1.0 
        # 4.075 -> 3.748
        # после буста метрика ухудшилась. Плохой буст, плохой. Вернем идею на место.
        factor = 0.0 
        # 4.075 -> 3.520
        # стало ещё хуже, вернём взад.
        factor = 1.0 
        # dcg@k 4.075 -> 3.748 - хороший такой буст. Отрицательный рост.
        if behaviour is not None:
            factor += math.log(
                add + mult * behaviour.exact + behaviour.complement
            )

        # Создаём новую запись, чтобы не менять исходную выдачу.
        reranked.append({
            **item,
            "final_score": item["bm25_score"] * factor,
        })

    return sorted(  # пока не понял, зачем сортировать
        reranked,
        key=lambda item: item["final_score"],
        reverse=True,
    )

def evaluate_bm25(
    index: BM25Index,
    part: DatasetPart,
    doc2behaviour: dict[str, BehaviourFeatures],
    k: int = 15,
) -> dict[str, float]:
    '''
    Должен вернуть, как минимум, {"dcg@k": ...}
    '''
    # YOUR CODE
    total_dcg = 0.0

    for query_id, query_text in part.queries.items():
        serp = index.search(query_text, top_k=k)

        # При пустом словаре оцениваем обычный BM25.
        if doc2behaviour:
            serp = rerank_with_behaviour(serp, doc2behaviour)

        # Переводим метки документов в значения релевантности.
        doc_ids = [item["doc_id"] for item in serp]
        labels = part.query2doc2target.get(query_id, {})
        gains = gains_for_query(doc_ids, labels)
        total_dcg += dcg_at_k(gains, k=k)

    # Усредняем метрику по запросам.
    n_queries = len(part.queries)
    return {"dcg@k": total_dcg / n_queries if n_queries else 0.0}
# ---- Конец кода ----

In [33]:
# Ассерт-тест к задаче 4

toy_bm25_serp = [
    {"doc_id": "d1", "bm25_score": 10.0},
    {"doc_id": "d2", "bm25_score": 9.0},
]
toy_beh = {
    "d1": BehaviourFeatures(exact=0, complement=0, substitute=0, exact_or_complement=0),
    "d2": BehaviourFeatures(exact=10, complement=0, substitute=0, exact_or_complement=10),
}
reranked = rerank_with_behaviour(toy_bm25_serp, toy_beh, add=1.0, mult=2.0)
assert reranked[0]["doc_id"] == "d2"  # d2 получает сильный буст

# Мини-оценка на игрушечных данных
toy_docs = {
    "d1": TextFields(product_title="apple iphone", product_text="smartphone"),
    "d2": TextFields(product_title="apple case", product_text="phone case"),
    "d3": TextFields(product_title="banana", product_text="fruit"),
}
toy_index = build_bm25_index(toy_docs, doc_ids=["d1","d2","d3"], fields=["product_title","product_text"])
toy_part = DatasetPart(
    queries={"q": "apple phone"},
    query2doc2target={"q": {"d1": "Exact", "d2": "Complement"}},
)
m = evaluate_bm25(toy_index, toy_part, doc2behaviour={}, k=3)
assert "dcg@k" in m and m["dcg@k"] > 0
print("✅ Task 4 OK")


✅ Task 4 OK


In [34]:
# Считаем качество BM25 (без поведенческого буста и с бустом)

bm25_plain = evaluate_bm25(bm25_index, test_part, doc2behaviour={}, k=15)
bm25_boosted = evaluate_bm25(bm25_index, test_part, doc2behaviour=indexed_plus_train_behaviour, k=15)

print("BM25 plain  :", bm25_plain)
print("BM25 boosted:", bm25_boosted)


BM25 plain  : {'dcg@k': 4.0750591543761185}
BM25 boosted: {'dcg@k': 3.748260840659483}


## Бонусный трек: OpenSearch Query DSL (необязательно, +2 бонусных балла)

Этот трек возвращает «инфраструктурную» сложность из исходной версии ДЗ:  
поднять **OpenSearch** в Docker, описать **mappings**, проиндексировать документы и собрать запрос с **function_score**.

**Связь с лекцией:**  
- обратный индекс и полнотекстовый поиск (**слайды 21–23**)  
- BM25 и параметры similarity (**слайды 27–39**)  
- query DSL и бустинг дополнительными сигналами (обсуждение `function_score`) (**слайды 43–48**)  

### Что нужно сделать
1) Поднимите OpenSearch в Docker (локально или в Colab, если получится).  
2) Реализуйте `get_mappings_opensearch(...)` и `describe_document_opensearch(...)`.  
3) Реализуйте `build_opensearch_query(...)`, чтобы:
   - искать по текстовым полям,
   - добавлять буст за счёт `behaviour.exact` через function_score.
4) Посчитайте nDCG@10 на `test_part` и сравните с вашей Colab-friendly реализацией (BM25Okapi).

### Критерий успеха
- Индекс создаётся, документы индексируются, запрос возвращает выдачу.
- Ваш буст поведенческим сигналом **улучшает** nDCG@10 хотя бы на небольшой величине
  (или вы показываете, почему не улучшает).

> Бонус сдаётся **по желанию**. Если Docker не поднимается в Colab — это нормально: делайте локально.


In [ ]:
!docker pull opensearchproject/opensearch:latest

In [ ]:
!docker run -d --rm --name opensearch -p 9200:9200 -p 9600:9600 -e "discovery.type=single-node" -e "plugins.security.ssl.http.enabled=false" -e OPENSEARCH_INITIAL_ADMIN_PASSWORD opensearchproject/opensearch:latest

In [36]:
# --- (опционально) Поднятие OpenSearch в Docker ---
# В Colab может не работать. Локально — должно.

# 1) Убедитесь, что docker доступен:
# !docker --version

# 2) Поднимаем OpenSearch (одной нодой)
import os, time, json
os.environ["OPENSEARCH_INITIAL_ADMIN_PASSWORD"] = "Deepschool4Soul"

# Локально (в терминале) или в Colab (если docker работает):
# export OPENSEARCH_INITIAL_ADMIN_PASSWORD=Deepschool4Soul
# !docker pull opensearchproject/opensearch:latest
# !docker run -d --rm --name opensearch \
#   -p 9200:9200 -p 9600:9600 \
#   -e "discovery.type=single-node" \
#   -e "plugins.security.ssl.http.enabled=false" \
#   -e OPENSEARCH_INITIAL_ADMIN_PASSWORD \
#   opensearchproject/opensearch:latest

# 3) Проверка доступности:
import requests
try:
    r = requests.get("http://localhost:9200", timeout=3, auth=('admin', os.environ["OPENSEARCH_INITIAL_ADMIN_PASSWORD"]))
    print("OpenSearch status:", r.status_code)
except Exception as e:
    print("OpenSearch недоступен (это ок, если вы не делаете бонус):", e)


OpenSearch недоступен (это ок, если вы не делаете бонус): HTTPConnectionPool(host='localhost', port=9200): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000001AFF2134AD0>: Failed to establish a new connection: [WinError 10061] Подключение не установлено, т.к. конечный компьютер отверг запрос на подключение'))


### Бонус-задача: Mappings и запрос (заполняйте только если делаете бонус)

Ниже — заглушки, которые нужно заполнить.  
Если OpenSearch не поднят, ячейки можно **не выполнять**.


In [ ]:
# --- Бонус: заглушки функций (OpenSearch) ---

from typing import Dict, Any

def get_mappings_opensearch() -> Dict[str, Any]:
    '''
    Верните mappings (и при желании settings/similarity) для индекса:
    - текстовые поля (title, description) как text (BM25)
    - поведенческие признаки как numeric (integer/float)
    '''
    # ---- Ваш код здесь ----
    raise NotImplementedError
    # ---- Конец кода ----


def describe_document_opensearch(doc_id: str, title: str, description: str, behaviour: dict) -> Dict[str, Any]:
    '''
    Соберите документ под ваш mappings. Минимально нужно:
    - product_id/doc_id
    - title, description
    - behaviour.exact (и/или другие счётчики)
    '''
    # ---- Ваш код здесь ----
    raise NotImplementedError
    # ---- Конец кода ----


def build_opensearch_query(query_text: str, k: int = 10) -> Dict[str, Any]:
    '''
    Query DSL:
    - полнотекстовый матч по title/description
    - function_score: буст за счёт behaviour.exact (монотонно)
    '''
    # ---- Ваш код здесь ----
    raise NotImplementedError
    # ---- Конец кода ----

## Часть 2. Векторный поиск (Qdrant в embedded-режиме)

Мы используем `QdrantClient(location=':memory:')` — это упрощённый режим без docker/серверов, который отлично подходит для обучения и Colab.

**Связь с лекцией:** векторный поиск (**слайды 41–42**).

# Задача 5: Получение эмбеддингов документов – 2 балла

**Зачем:** чтобы сделать vector-search, нужно превратить текст в вектор и выбрать пространство/метрику (у нас будет cosine).

### Что нужно сделать
1) Реализуйте `obtain_embeddings(texts)` — батчево посчитать эмбеддинги.
2) Реализуйте `obtain_doc2embedding(text_rich_docs, doc_ids, field)` — получить словарь `doc_id -> vector`
   для выбранного поля (`product_title` или `product_text`).


In [55]:
from sentence_transformers import SentenceTransformer

# Небольшой и быстрый эмбеддер (подходит под Colab CPU/GPU)
EMB_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMB_MODEL_NAME)

# ---- Ваш код здесь ----
def obtain_embeddings(texts: list[str], batch_size: int = 64) -> np.ndarray:
    '''
    Return: np.ndarray of shape [len(texts), dim]
    '''
    # YOUR CODE
    # Для пустого списка сохраняем форму результата [0, dim].
    if not texts:
        dim = embedder.get_sentence_embedding_dimension()
        return np.empty((0, dim), dtype=np.float32)

    # Считаем векторы батчами и нормализуем их до единичной длины.
    embeddings = embedder.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        convert_to_tensor=False,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    # Явно обеспечиваем тип результата np.ndarray.
    return np.asarray(embeddings, dtype=np.float32)

def obtain_doc2embedding(
    text_rich_docs: dict[str, TextFields],
    doc_ids: list[str],
    field: str,
    batch_size: int = 64,
) -> dict[str, np.ndarray]:
    '''
    field in {"product_title", "product_text"}
    '''
    # YOUR CODE
    # Собираем выбранное поле в порядке идентификаторов документов.
    texts = [
        getattr(text_rich_docs[doc_id], field)
        for doc_id in doc_ids
    ]
    embeddings = obtain_embeddings(texts, batch_size=batch_size)

    # Сопоставляем каждому документу соответствующий вектор.
    return {
        doc_id: vector
        for doc_id, vector in zip(doc_ids, embeddings)
    }
# ---- Конец кода ----

In [56]:
# Ассерт-тест к задаче 5

vecs = obtain_embeddings(["hello world", "deep learning"], batch_size=2)
assert isinstance(vecs, np.ndarray) and vecs.ndim == 2 and vecs.shape[0] == 2
assert np.allclose(np.linalg.norm(vecs, axis=1), 1.0, atol=1e-3)  # normalize_embeddings=True

doc2v = obtain_doc2embedding(
    {"d": TextFields(product_title="a", product_text="b")},
    doc_ids=["d"],
    field="product_title",
    batch_size=1
)
assert "d" in doc2v and doc2v["d"].shape == (vecs.shape[1],)
print("✅ Task 5 OK")


✅ Task 5 OK


In [57]:
# Считаем эмбеддинги для каталога документов (2 набора: title и text)

doc2emb_title = obtain_doc2embedding(text_rich_docs, ALL_DOC_IDS, field="product_title", batch_size=64)
doc2emb_text  = obtain_doc2embedding(text_rich_docs, ALL_DOC_IDS, field="product_text", batch_size=64)

# sanity
dim = next(iter(doc2emb_title.values())).shape[0]
assert all(v.shape == (dim,) for v in doc2emb_title.values())
assert all(v.shape == (dim,) for v in doc2emb_text.values())

print("Embedding dim:", dim)


Embedding dim: 384


# Задача 6: Индексация в Qdrant – 5 баллов

**Зачем:** Qdrant хранит векторы и быстро ищет ближайших соседей.
Мы положим 2 представления одного документа в **named vectors**:
- `title_vec`
- `text_vec`

### Что нужно сделать
1) Реализуйте `create_qdrant_collection(client, collection_name, dim)` с двумя named vectors и cosine distance.
2) Реализуйте `upload_to_qdrant(...)`, чтобы залить все документы (id + payload + vectors).


In [58]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from uuid_utils import NAMESPACE_URL, uuid5

COLLECTION = "products"

# ---- Ваш код здесь ----
def create_qdrant_collection(client: QdrantClient, collection_name: str, dim: int) -> None:
    # YOUR CODE
    # При повторном запуске сохраняем существующую коллекцию.
    if client.collection_exists(collection_name):
        return

    # Для каждого товара храним два вектора с косинусной метрикой.
    client.create_collection(
        collection_name=collection_name,
        vectors_config={
            "title_vec": VectorParams(size=dim, distance=Distance.COSINE),
            "text_vec": VectorParams(size=dim, distance=Distance.COSINE),
        },
    )

def upload_to_qdrant(
    client: QdrantClient,
    collection_name: str,
    doc_ids: list[str],
    doc2emb_title: dict[str, np.ndarray],
    doc2emb_text: dict[str, np.ndarray],
    batch_size: int = 256,
) -> None:
    # YOUR CODE
    if batch_size <= 0:
        raise ValueError("batch_size должен быть положительным")

    # Формируем и загружаем записи небольшими порциями.
    for start in range(0, len(doc_ids), batch_size):
        points = []

        for doc_id in doc_ids[start:start + batch_size]:
            # Из исходного doc_id получаем постоянный UUID. 
            # Можно было и номер документа, но УУИД сохранится при выборе других срезов или другой сортировке.
            point_id = str(uuid5(NAMESPACE_URL, f"product:{doc_id}"))

            points.append(
                PointStruct(
                    id=point_id,
                    vector={
                        "title_vec": doc2emb_title[doc_id].tolist(),
                        "text_vec": doc2emb_text[doc_id].tolist(),
                    },
                    payload={"doc_id": doc_id},
                )
            )

        # Добавляем новые товары или обновляем уже существующие.
        client.upsert(
            collection_name=collection_name,
            points=points,
            wait=True,
        )

# ---- Конец кода ----

In [59]:
# Ассерт-тест к задаче 6

client = QdrantClient(location=":memory:")
create_qdrant_collection(client, "toy", dim=4)

doc_ids = ["a","b","c"]
doc2t = {d: np.random.randn(4).astype(np.float32) for d in doc_ids}
doc2x = {d: np.random.randn(4).astype(np.float32) for d in doc_ids}
upload_to_qdrant(client, "toy", doc_ids, doc2t, doc2x, batch_size=2)

info = client.get_collection("toy")
assert info.points_count == 3
print("✅ Task 6 OK")


✅ Task 6 OK


In [60]:
# Поднимаем embedded-Qdrant и заливаем документы

qdrant = QdrantClient(location=":memory:")
create_qdrant_collection(qdrant, COLLECTION, dim=dim)
upload_to_qdrant(qdrant, COLLECTION, ALL_DOC_IDS, doc2emb_title, doc2emb_text, batch_size=256)

print("Qdrant points:", qdrant.get_collection(COLLECTION).points_count)


Qdrant points: 1683


# Задача 7: Отбор кандидатов (multi-vector) – 8 баллов

**Зачем:** у нас есть два «источника кандидатов»:
- поиск по `title_vec`
- поиск по `text_vec`

В гибридных системах это типичная история: *несколько retrieval-каналов → объединение → rerank.*

**Связь с лекцией:** гибридный поиск (**слайд 48**).

### Что нужно сделать
1) Реализуйте `qdrant_search(...)` (обёртка над клиентом).
2) Реализуйте `merge_serps(...)`, которая:
   - принимает 2 выдачи (title/text),
   - делает union по `doc_id`,
   - сохраняет обе оценки (если документа не было в источнике — ставим 0),
   - возвращает список словарей и **не теряет сортировку** (сортируем уже по комбинированному скору дальше).

3) Реализуйте `retrieve_vector_candidates(...)`, которая для текста запроса получает кандидатов из двух каналов.


In [61]:
# ---- Ваш код здесь ----
def qdrant_search(
    client: QdrantClient,
    collection_name: str,
    query_vec: np.ndarray,
    vector_name: str,
    top_k: int = 15,
) -> list[dict[str, T.Any]]:
    # YOUR CODE
    if top_k <= 0:
        return []   # а то ругнется Qdrant на отрицательный лимит

    # Указываем, среди каких векторов искать: title_vec или text_vec.
    # Вектор запроса преобразуем из NumPy-массива в обычный список.
    hits = client.search(   # вот интересно, а если исходный вектор не нормализован, то Qdrant сам его нормализует?
        collection_name=collection_name,
        query_vector=(vector_name, query_vec.tolist()),
        limit=top_k,
        with_payload=True,
        with_vectors=False,
    )

    serp = []
    for hit in hits:
        # Исходный doc_id хранится в payload.
        # hit.id — внутренний UUID записи, его здесь не используем.
        payload = hit.payload
        if payload is None or "doc_id" not in payload:
            raise ValueError(f"У записи {hit.id} отсутствует doc_id")

        serp.append({
            "doc_id": payload["doc_id"],
            "score": float(hit.score),
        })

    # Сохраняем порядок результатов, полученный от Qdrant.
    return serp

def merge_serps(
    title_serp: list[dict[str, T.Any]],
    text_serp: list[dict[str, T.Any]],
) -> list[dict[str, T.Any]]:
    '''
    Каждый элемент выдачи должен иметь минимум:
      - doc_id
      - score

    Возвращаем список:
      {"doc_id": ..., "score_title": ..., "score_text": ...}
    '''
    # YOUR CODE
    # dict побыстрее будет искать по doc_id, а порядок добавления Python сохранит.
    merged = {}

    for item in title_serp:
        doc_id = item["doc_id"]

        # Сначала добавляем документы в порядке выдачи по названию.
        merged[doc_id] = {
            "doc_id": doc_id,
            "score_title": item["score"],
            "score_text": 0.0,
        }

    for item in text_serp:
        doc_id = item["doc_id"]

        if doc_id not in merged:
            # Документ найден только по описанию:
            # добавляем его в конец с нулевой оценкой по названию.
            merged[doc_id] = {
                "doc_id": doc_id,
                "score_title": 0.0,
                "score_text": 0.0,
            }

        # Для общего документа дополняем существующую запись.
        # Его положение в словаре при этом не меняется.
        merged[doc_id]["score_text"] = item["score"]

    # Итоговую сортировку пока не делаем, оставляем в порядке появления документов в title_serp, а затем в text_serp.
    return list(merged.values())

def retrieve_vector_candidates(
    client: QdrantClient,
    collection_name: str,
    query_text: str,
    top_k: int = 15,
) -> list[dict[str, T.Any]]:
    # YOUR CODE
    if top_k <= 0:
        return []

    # Функция принимает список текстов и возвращает матрицу.
    # Передаём один запрос и берём первую строку — его вектор.
    query_vec = obtain_embeddings([query_text], batch_size=1)[0]

    # Один и тот же вектор запроса сравниваем отдельно
    # с векторами названий и с векторами описаний товаров.
    title_serp = qdrant_search(
        client,
        collection_name,
        query_vec,
        vector_name="title_vec",
        top_k=top_k,
    )
    text_serp = qdrant_search(
        client,
        collection_name,
        query_vec,
        vector_name="text_vec",
        top_k=top_k,
    )

    # Объединяем кандидатов без дубликатов, сохраняя обе оценки.
    return merge_serps(title_serp, text_serp)
# ---- Конец кода ----

In [62]:
# Ассерт-тест к задаче 7 (без настоящего Qdrant)

t = [{"doc_id": "d1", "score": 0.9}, {"doc_id": "d2", "score": 0.8}]
x = [{"doc_id": "d2", "score": 0.7}, {"doc_id": "d3", "score": 0.6}]
m = merge_serps(t, x)

m_by = {r["doc_id"]: r for r in m}
assert set(m_by) == {"d1","d2","d3"}
assert m_by["d1"]["score_title"] == 0.9 and m_by["d1"]["score_text"] == 0.0
assert m_by["d2"]["score_title"] == 0.8 and m_by["d2"]["score_text"] == 0.7
assert m_by["d3"]["score_title"] == 0.0 and m_by["d3"]["score_text"] == 0.6
print("✅ Task 7 OK")


✅ Task 7 OK


In [63]:
# Пример: посмотрим на кандидатов по одному тестовому запросу (не оценивается)

_any_qid, _any_qtext = next(iter(test_part.queries.items()))
cands = retrieve_vector_candidates(qdrant, COLLECTION, _any_qtext, top_k=5)
print("Query:", _any_qtext)
print("Top candidates:", cands[:3])

# надо посмотреть на глаз, что там реально лежит
for cand in cands[:3]:
    doc_id = cand["doc_id"]
    document = text_rich_docs[doc_id]

    print(f"\n--- doc_id: {doc_id} ---")
    print("Название:", document.product_title)
    print("Текст:", document.product_text)

# Вот оно как. А я думал - вентиляторы.


Query: #4 pads without wings
Top candidates: [{'doc_id': 'B01IAI9WNO', 'score_title': 0.7182208946519171, 'score_text': 0.606077825050024}, {'doc_id': 'B01IAI9UZ4', 'score_title': 0.71329536632772, 'score_text': 0.0}, {'doc_id': 'B00OJPBESO', 'score_title': 0.7035854436977477, 'score_text': 0.6662749370996505}]

--- doc_id: B01IAI9WNO ---
Название: Always Ultra Thin Pads without Wings, Long Super 20 ea (Pack of 4)
Текст: Always Ultra Thin Pads without Wings, Long Super 20 ea (Pack of 4)
Always
None
Always Ultra Thin Sanitary Pads ideal for overnight use because they are designed especially for when you are lying down.
None

--- doc_id: B01IAI9UZ4 ---
Название: Always Ultra Thin Pads without Wings, Long Super 20 ea (Pack of 3)
Текст: Always Ultra Thin Pads without Wings, Long Super 20 ea (Pack of 3)
Always
None
Always Ultra Thin Sanitary Pads ideal for overnight use because they are designed especially for when you are lying down.
None

--- doc_id: B00OJPBESO ---
Название: Always Ultra 

# Задача 8: Оценка качества векторного поиска – 5 баллов

**Зачем:** vector-search часто даёт более «семантические» кандидаты, но может уступать BM25 на точных запросах.
Мы оценим качество отдельно.

**Связь с лекцией:** сравнение подходов + метрики (**слайды 13–16**, **41–42**).

### Что нужно сделать
1) Реализуйте `score_vector_candidates(...)` — вычислить итоговый скор и отсортировать выдачу.
2) Реализуйте `evaluate_vector(...)` — посчитать средний `DCG@K` на `test_part`.


In [64]:
# ---- Ваш код здесь ----
def score_vector_candidates(
    candidates: list[dict[str, T.Any]],
    w_title: float = 0.5,
    w_text: float = 0.5,
) -> list[dict[str, T.Any]]:
    '''
    candidates: [{"doc_id", "score_title", "score_text"}, ...]
    Выход: тот же список + ключ "final_score", отсортированный по нему (desc).
    '''
    # YOUR CODE
    scored = []

    for candidate in candidates:
        # Объединяем оценки названия и описания с заданными весами.
        final_score = (
            w_title * candidate["score_title"]
            + w_text * candidate["score_text"]
        )

        # Создаём новую запись, сохраняя исходные оценки.
        # Входной список и его словари не изменяем.
        scored.append({
            **candidate,
            "final_score": final_score,
        })

    # Более высокий итоговый скор соответствует лучшей позиции.
    return sorted(
        scored,
        key=lambda item: item["final_score"],
        reverse=True,
    )

def evaluate_vector(
    client: QdrantClient,
    collection_name: str,
    part: DatasetPart,
    k: int = 15,
    w_title: float = 0.5,
    w_text: float = 0.5,
) -> dict[str, float]:
    # YOUR CODE
    total_dcg = 0.0

    for query_id, query_text in part.queries.items():
        # Получаем до k кандидатов из каждого канала.
        # После объединения их может быть до 2 * k.
        candidates = retrieve_vector_candidates(
            client,
            collection_name,
            query_text,
            top_k=k,
        )

        # Сортируем объединённую выдачу по взвешенной сумме оценок.
        ranked = score_vector_candidates(
            candidates,
            w_title=w_title,
            w_text=w_text,
        )

        # Для оценки оставляем только первые k документов.
        doc_ids = [item["doc_id"] for item in ranked[:k]]

        # Получаем релевантность документов из разметки запроса.
        labels = part.query2doc2target.get(query_id, {})
        gains = gains_for_query(doc_ids, labels)
        total_dcg += dcg_at_k(gains, k=k)

    # Средний DCG по всем запросам; для пустой части возвращаем 0.
    n_queries = len(part.queries)
    return {"dcg@k": total_dcg / n_queries if n_queries else 0.0}

# ---- Конец кода ----

In [65]:
# Ассерт-тест к задаче 8

c = [
    {"doc_id": "d1", "score_title": 0.8, "score_text": 0.0},
    {"doc_id": "d2", "score_title": 0.1, "score_text": 0.8},
]
s = score_vector_candidates(c, w_title=0.5, w_text=0.5)
rprint(s)
# вот с какого перепугу 0.05? Заголовок-то соответствует.

assert s[0]["doc_id"] == "d2"  # 0.45 vs 0.05
assert "final_score" in s[0]
print("✅ Task 8 OK")


[
    {'doc_id': 'd2', 'score_title': 0.1, 'score_text': 0.8, 'final_score': 0.45},
    {'doc_id': 'd1', 'score_title': 0.8, 'score_text': 0.0, 'final_score': 0.4}
]

✅ Task 8 OK


In [66]:
# Считаем качество vector-search

vector_metrics = evaluate_vector(qdrant, COLLECTION, test_part, k=15, w_title=0.5, w_text=0.5)
print("Vector metrics:", vector_metrics)

# Ура! Лучше, чем BM25 получилось! Вот что вектор животворящий делает.


Vector metrics: {'dcg@k': 4.180288041966783}


## Часть 3. Гибридный поиск и простое переранжирование

В реальных системах часто используют **двухэтапный пайплайн**:
1) retrieval (быстро): BM25 и/или vector-search  
2) rerank (точнее, но дороже): модель на признаках кандидатов

**Связь с лекцией:** гибридный поиск и компромиссы (**слайд 48**), метрики (**слайды 13–16**).

### Важно про таргет-лик
- Поведенческие признаки для reranker берём **ТОЛЬКО из `indexed_part`** (это «прошлые логи»).
- Нельзя считать `BehaviourFeatures` по `train_part` и потом использовать их как фичи на этих же запросах — это прямой ликомет.


# Задача 9: Обучение reranker – 8 баллов

**Цель:** обучить модель, которая по набору признаков кандидата (BM25 + векторные скоры + поведенческие фичи)
предсказывает релевантность и переупорядочивает кандидатов.

**Связь с лекцией:** почему гибрид работает лучше одного канала (**слайд 48**).

### Что нужно сделать (пошагово)
1) Для каждого запроса из `train_part` получите кандидатов:
   - topK из BM25 (`bm25_index.search`)
   - topK из Qdrant (`retrieve_vector_candidates`)
   - объедините кандидатов по `doc_id`.

2) Для каждого `(query_id, doc_id)` соберите признаки:
   - `bm25_score`
   - `score_title`, `score_text`
   - поведенческие: `exact`, `complement`, `substitute`, `exact_or_complement` (**только из `indexed_behaviour`**)

3) Соберите обучающую таблицу и обучите `CatBoostRanker`:
   - `group_id` = `query_id`
   - `target` = `gain` по `LABEL2GAIN`

4) Напишите функцию `predict_rerank(...)`, которая выдаёт отсортированную выдачу по предикту модели.


In [67]:
import pandas as pd
from catboost import CatBoostRanker, Pool

def _bm25_candidates(query_text: str, top_k: int = 30) -> dict[str, float]:
    serp = bm25_index.search(query_text, top_k=top_k)
    return {row["doc_id"]: float(row["bm25_score"]) for row in serp}

def _vector_candidates(query_text: str, top_k: int = 30) -> dict[str, dict[str, float]]:
    cands = retrieve_vector_candidates(qdrant, COLLECTION, query_text, top_k=top_k)
    return {row["doc_id"]: {"score_title": float(row["score_title"]), "score_text": float(row["score_text"])} for row in cands}

def _join_candidates(
    bm25: dict[str, float],
    vec: dict[str, dict[str, float]],
) -> list[dict[str, T.Any]]:
    doc_ids = set(bm25) | set(vec)
    rows = []
    for d in doc_ids:
        rows.append({
            "doc_id": d,
            "bm25_score": bm25.get(d, 0.0),
            "score_title": vec.get(d, {}).get("score_title", 0.0),
            "score_text": vec.get(d, {}).get("score_text", 0.0),
        })
    return rows

# ---- Ваш код здесь ----
def build_rerank_dataframe(
    part: DatasetPart,
    indexed_behaviour: dict[str, BehaviourFeatures],
    top_k_bm25: int = 30,
    top_k_vec: int = 30,
) -> tuple[pd.DataFrame, list[str]]:
    '''
    Возвращает:
      - df со столбцами: query_id, doc_id, target_gain, + feature columns
      - список feature columns (для обучения)
    '''
    # YOUR CODE
    # семь признаков из условия
    feature_cols = [
        "bm25_score",
        "score_title",
        "score_text",
        "exact",
        "complement",
        "substitute",
        "exact_or_complement",
    ]
    rows = []

    # по каждому запросу получаем кандидатов из двух каналов, объединяем их и формируем обучающую таблицу. По шагам.
    for query_id, query_text in part.queries.items():

        # Шаг 1. Для каждого запроса получаем кандидатов из двух каналов.
        bm25 = _bm25_candidates(query_text, top_k=top_k_bm25)
        vec = _vector_candidates(query_text, top_k=top_k_vec)

        # Объединяем по doc_id, сохраняя оценки обоих каналов.
        candidates = _join_candidates(bm25, vec)
        labels = part.query2doc2target.get(query_id, {})

        for candidate in candidates:
            doc_id = candidate["doc_id"]

            # Шаг 2. Поведенческие признаки берём только из indexed.
            # Для неизвестного документа все счётчики будут нулевыми.
            behaviour = indexed_behaviour.get(doc_id)
            if behaviour is None:
                behaviour = BehaviourFeatures()

            # Шаг 3. Формируем строку обучающей таблицы:
            # идентификаторы, целевая релевантность и семь признаков.
            rows.append({
                "query_id": query_id,
                "doc_id": doc_id,
                "target_gain": float(
                    LABEL2GAIN.get(labels.get(doc_id), 0.0)
                ),
                "bm25_score": candidate["bm25_score"],
                "score_title": candidate["score_title"],
                "score_text": candidate["score_text"],
                "exact": behaviour.exact,
                "complement": behaviour.complement,
                "substitute": behaviour.substitute,
                "exact_or_complement": behaviour.exact_or_complement,
            })

    # Явно задаём столбцы, чтобы они были даже у пустой таблицы.
    columns = ["query_id", "doc_id", "target_gain"] + feature_cols
    return pd.DataFrame(rows, columns=columns), feature_cols


def train_reranker(
    df: pd.DataFrame,
    feature_cols: list[str],
) -> CatBoostRanker:
    # YOUR CODE
    if df.empty:
        raise ValueError("Обучающая таблица пуста")

    # Шаг 3 (второй полушаг). Строки одного запроса должны идти подряд для CatBoostRanker, поэтому отсортируем.
    train_df = df.sort_values(
        "query_id", kind="stable"
    ).reset_index(drop=True)

    # query_id задаёт группу, внутри которой учимся сравнивать товары.
    # Идентификаторы и target_gain не включаем во входные признаки.
    train_pool = Pool(
        data=train_df[feature_cols],
        label=train_df["target_gain"],
        group_id=train_df["query_id"].astype(str).tolist(),
    )

    # Обучаем модель ранжирования с начальными параметрами.
    model = CatBoostRanker(
        loss_function="YetiRank",
        iterations=200,
        depth=6,
        learning_rate=0.05,
        random_seed=SEED,
        verbose=50,
        allow_writing_files=False,
    )
    model.fit(train_pool)
    return model

def predict_rerank(
    model: CatBoostRanker,
    query_text: str,
    indexed_behaviour: dict[str, BehaviourFeatures],
    top_k_bm25: int = 30,
    top_k_vec: int = 30,
    final_k: int = 15,
) -> list[str]:
    '''
    Возвращает список doc_id длины final_k в порядке убывания релевантности.
    '''
    # YOUR CODE
    if final_k <= 0:
        return []

    # Шаг 4. Готовим один запрос без целевых меток.
    # Опять через build_rerank_dataframe, чтобы признаки при обучении
    # и предсказании вычислялись единообразно.
    query_part = DatasetPart(
        queries={"prediction": query_text},
        query2doc2target={},
    )
    df, _ = build_rerank_dataframe(
        query_part,
        indexed_behaviour=indexed_behaviour,
        top_k_bm25=top_k_bm25,
        top_k_vec=top_k_vec,
    )

    if df.empty:
        return []

    # Берём признаки в том же порядке, в котором обучалась модель.
    # target_gain здесь не используется.
    scores = model.predict(df[model.feature_names_])

    # Сортируем по предсказанному скору, от большего к меньшему.
    ranked = df.assign(final_score=scores).sort_values(
        "final_score",
        ascending=False,
        kind="stable",
    )

    # Возвращаем первые final_k идентификаторов документов - самые релевантные, как мы надеемся.
    return ranked["doc_id"].head(final_k).tolist()

# ---- Конец кода ----

In [68]:
# Ассерт-тест к задаче 9 (проверяем контракт данных)

df_train, feat_cols = build_rerank_dataframe(_subsample_part(train_part, 5, seed=SEED+10), indexed_behaviour=indexed_behaviour, top_k_bm25=5, top_k_vec=5)

assert len(df_train) > 0
assert set(["query_id","doc_id","target_gain"]).issubset(df_train.columns)
assert all(c in df_train.columns for c in feat_cols)
assert df_train.groupby("query_id").size().min() > 0

print("✅ Task 9 OK (data contract)")


✅ Task 9 OK (data contract)


In [69]:
# Обучаем reranker на train_part (можно уменьшить top_k для ускорения экспериментов)

df_train, feature_cols = build_rerank_dataframe(train_part, indexed_behaviour=indexed_behaviour, top_k_bm25=30, top_k_vec=30)
print("Train rows:", len(df_train))

reranker = train_reranker(df_train, feature_cols)


Train rows: 25264
0:	total: 6.51ms	remaining: 1.29s
50:	total: 237ms	remaining: 694ms
100:	total: 459ms	remaining: 450ms
150:	total: 678ms	remaining: 220ms
199:	total: 892ms	remaining: 0us


# Задача 10: Финальная оценка гибридной системы – 2 балла

**Зачем:** инженерное решение — это всегда сравнение с базовыми линиями и понимание компромисса.

**Связь с лекцией:** метрики и сравнение подходов (**слайды 13–16**, **41–42**, **48**).

### Что нужно сделать
1) Реализуйте `evaluate_reranker(...)` и посчитайте `DCG@K` на `test_part`.
2) Сравните три числа:
   - BM25 (plain / boosted),
   - vector-search,
   - hybrid+rerank.

В конце ноутбука оставьте короткий вывод: **какой сигнал дал прирост и почему** (2–5 предложений).


In [71]:
# ---- Ваш код здесь ----
def evaluate_reranker(
    model: CatBoostRanker,
    part: DatasetPart,
    indexed_behaviour: dict[str, BehaviourFeatures],
    k: int = 15,
) -> dict[str, float]:
    # YOUR CODE
    total_dcg = 0.0

    # Для каждого тестового запроса получаем гибридную выдачу.
    for query_id, query_text in part.queries.items():
        doc_ids = predict_rerank(
            model,
            query_text,
            indexed_behaviour=indexed_behaviour,
            top_k_bm25=30,
            top_k_vec=30,
            final_k=k,
        )

        # Поведенческие признаки берём только из indexed.
        # Тестовые метки используем исключительно для оценки выдачи.
        labels = part.query2doc2target.get(query_id, {})
        gains = gains_for_query(doc_ids, labels)
        total_dcg += dcg_at_k(gains, k=k)

    # Усредняем DCG по тестовым запросам.
    n_queries = len(part.queries)
    return {"dcg@k": total_dcg / n_queries if n_queries else 0.0}

# ---- Конец кода ----

In [72]:
# Ассерт-тест к задаче 10

m = evaluate_reranker(reranker, _subsample_part(test_part, 10, seed=SEED+123), indexed_behaviour=indexed_behaviour, k=10)
assert "dcg@k" in m and np.isfinite(m["dcg@k"])
print("✅ Task 10 OK")


✅ Task 10 OK


In [73]:
# Финальное сравнение

hybrid_metrics = evaluate_reranker(reranker, test_part, indexed_behaviour=indexed_behaviour, k=15)

print("BM25 plain  :", bm25_plain)
print("BM25 boosted:", bm25_boosted)
print("Vector      :", vector_metrics)
print("Hybrid+rank :", hybrid_metrics)


BM25 plain  : {'dcg@k': 4.0750591543761185}
BM25 boosted: {'dcg@k': 3.748260840659483}
Vector      : {'dcg@k': 4.180288041966783}
Hybrid+rank : {'dcg@k': 4.522118075259733}


## Бонус (вне зачёта): адаптивный гибридный скоринг

Попробуйте придумать **query-dependent** смешивание каналов retrieval, например:
- если BM25 даёт очень уверенный top-1 (большой разрыв скоринга) — доверяем BM25 больше;
- если запрос длинный/«разговорный» — доверяем векторному поиску больше.

Сделайте 2–3 гипотезы, прогоните на `test_part` и коротко объясните результат.


# Заключение по домашке

Ниже — короткая рефлексия, чтобы закрепить ключевые идеи лекции и связать их с тем, что вы сделали руками.

## Рефлексия (связь с лекцией)

Ответьте кратко (3–6 предложений на пункт):

1) Где **BM25** выигрывает у **vector-search** и почему? (слайды **27–39** vs **41–42**)  
**Ваш ответ:** …

2) Какие данные и разметка нужны для **переранжирования / L2R** и почему их сбор почти всегда дороже, чем обучение модели? (слайды **51–54**)  
**Ваш ответ:** …

3) Почему **reranker** обычно ставят вторым этапом (после candidate generation), а не ранжируют им весь корпус? (слайды **48**, **51**)  
**Ваш ответ:** …

## Что сделали

- Собрали небольшой корпус документов и набор запросов с релевантностями.
- Построили **лексический retrieval (BM25)** и **векторный retrieval**, сравнили качество.
- Сделали **candidate generation** и собрали признаки для переранжирования.
- Обучили **CatBoostRanker** и оценили, насколько reranker улучшает метрики.

## Чему научились (прямо по результатам)

- Понимать компромисс *точность ↔ скорость ↔ стоимость* между BM25, dense retrieval и гибридом.
- Проектировать двухэтапный пайплайн поиска: **retrieval → rerank** (и почему это стандарт в проде).
- Готовить данные для L2R так, чтобы эксперимент был воспроизводимым и сравнимым.

## Как применять дальше

- Добавить **калибровку/нормировку скорингов** и более устойчивые схемы гибрида (RRF, weighted sum с валидацией веса).
- Перейти от «учебного» reranker-а к production-пайплайну: логирование, A/B, анти-лик, регулярное переобучение.
- Подключить real-time сигналы (freshness, персонализация) и аккуратно валидировать, что они не ломают оффлайн-метрики.


```code
BM25 plain  : {'dcg@k': 4.0750591543761185}
BM25 boosted: {'dcg@k': 3.748260840659483}
Vector      : {'dcg@k': 4.180288041966783}
Hybrid+rank : {'dcg@k': 4.522118075259733}
```

Рост очевиден, а вот BM25 boosted не дал роста потому, я думаю, что поведенческие признаки были несколько "с потолка", я там это отметил.

1) Где **BM25** выигрывает у **vector-search** и почему? (слайды **27–39** vs **41–42**)  
**Ваш ответ:** 

2) Какие данные и разметка нужны для **переранжирования / L2R** и почему их сбор почти всегда дороже, чем обучение модели? (слайды **51–54**)  
**Ваш ответ:** Для обучения специализированной модели ранжирования нужны данные с адекватной разметкой релевантности для каждого запроса и документа. 
Это более сложная задача, чем обычная разметка, и требует ручной работы экспертов (или тяжелых LLM), что делает процесс дорогостоящим. 
Кроме того, необходимо обеспечить разнообразие запросов и документов, чтобы покрыть заведомо неточные запросы пользователей.

3) Почему **reranker** обычно ставят вторым этапом (после candidate generation), а не ранжируют им весь корпус? (слайды **48**, **51**)  
**Ваш ответ:** reranker существенно ресурсоемкий и по времени и по вычислениям, поэтому используют только для небольшого числа кандидатов, а не для всего корпуса.

